# 01 - Data Acquisition

## <b> ADRIAN VAZQUEZ </b>

---

## Phase 1: Data Acquisition

Before any statistical modeling can be performed, a reliable historical dataset must be constructed.

The objectives of this notebook are:

1. Download historical market metadata from Polymarket.
2. Identify markets related to:
   - Federal Reserve decisions
   - Interest rate changes
   - Inflation and CPI releases
   - Monetary policy events
3. Extract YES outcome token identifiers.
4. Download historical implied probabilities from the Polymarket CLOB.
5. Validate data coverage and storage structure.
6. Construct the first research dataset for subsequent analysis.

---

## Data Sources

### Gamma API

Used for market discovery and metadata collection.

Information collected:

- Market ID
- Question
- Resolution date
- Volume
- Liquidity
- Outcome definitions
- CLOB token identifiers

---

### CLOB API

Used for historical price retrieval.

Each YES token represents a binary contract whose price can be interpreted as the market-implied probability of the event occurring.

Historical probabilities are retrieved using:

```text
https://clob.polymarket.com/prices-history
```

---

## Engineering Challenges Discovered

During the acquisition process several issues were identified:

### 1. Historical markets are not directly accessible

The standard `prices-history` endpoint frequently returned empty datasets for resolved markets.

### 2. Long time windows are rejected

The API rejects requests covering large date ranges with the error:

```text
invalid filters: 'startTs' and 'endTs' interval is too long
```

### 3. Time-window chunking

To overcome API limitations, historical downloads were implemented using temporal chunking.

The requested interval is split into smaller windows and later reconstructed into a continuous time series.

This approach successfully recovered historical probability trajectories for multiple resolved prediction markets.

---

## Output of this Notebook

The final output is a collection of historical market probability series:

| timestamp | implied_probability | market_id |
|------------|------------|------------|
| 2024-03-21 | 0.16 | 254578 |
| 2024-03-21 | 0.17 | 254578 |
| ... | ... | ... |

along with market metadata:

| market_id | question | volume | liquidity | resolution_date |
|------------|------------|------------|------------|------------|

These datasets form the foundation for the next stage of the research process:

**Phase 2 — Exploratory Analysis and Market Calibration.**

---

## Key Result

A robust data acquisition pipeline was successfully developed, allowing the recovery of historical implied probabilities for resolved Polymarket markets.

This validates the feasibility of constructing a prediction-market research dataset suitable for Bayesian probability estimation, calibration analysis, and mispricing detection.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected+png" 

from IPython.display import Image, display
import sys
import os
# Add project root to path
sys.path.append(os.path.abspath(".."))
# libs 
import numpy as np
import plotly.graph_objects as go
import requests
import pandas as pd
import ast
import time
import importlib
# import data market from polymarket 
import src.analytics.download_data as download_data
importlib.reload(download_data)
from src.analytics.download_data import *

In [ ]:
# 1. Download closed markets
df_markets_raw = fetch_many_markets(max_pages=50,limit=500,closed=True)
print("Total markets download", len(df_markets_raw))

# 2. Filter Fed  - rates markets
df_fed = filter_fed_rates_markets(df_markets_raw)
df_fed["yes_token_id"] = df_fed.apply(extract_yes_token_id, axis=1)
cols_to_keep = [
    "id",
    "question",
    "slug",
    "category",
    "endDate",
    "closed",
    "volume",
    "liquidity",
    "outcomes",
    "outcomePrices",
    "clobTokenIds",
    "yes_token_id"
]
cols_to_keep = [c for c in cols_to_keep if c in df_fed.columns]

df_fed = df_fed[cols_to_keep].dropna(subset=["yes_token_id"]).copy()
df_fed["endDate_dt"] = pd.to_datetime(df_fed["endDate"],errors="coerce",utc=True)
df_fed["volume"] = pd.to_numeric(df_fed["volume"], errors="coerce")
df_fed["liquidity"] = pd.to_numeric(df_fed["liquidity"], errors="coerce")


# 3. Keep relevant recent markets
df_fed = df_fed[(df_fed["endDate_dt"] >= "2023-01-01") & (df_fed["volume"].fillna(0) > 0) & (df_fed["yes_token_id"].notna())].copy()
print("Fed-rates markets after validation:", len(df_fed))
display(df_fed[["id", "question", "endDate_dt", "volume", "liquidity", "yes_token_id"]].sort_values("endDate_dt", ascending=False).head(20))

# 4. Test one specific Fed market
try:
    market = df_fed[df_fed["question"].str.contains("Fed cut interest rates 4 times",case=False,na=False)].iloc[0]
    token_id = market["yes_token_id"]
    end_ts = int(market["endDate_dt"].timestamp())
    start_ts = int((market["endDate_dt"] - pd.Timedelta(days=365)).timestamp())

    # calling the 7 micro-chunks function
    hist = get_price_history_chunked(clob_token_id=token_id,start_ts=start_ts,end_ts=end_ts,fidelity_minutes=60,chunk_days=7)
    print("\nSelected market:")
    print(market["id"], market["question"])
    print("Price history shape:", hist.shape)

    if not hist.empty:
        display(hist.head())
    else:
        print("Warning: Completed chunks successfully but no trade history records were returned by the CLOB.")

except IndexError:
    print("Error:  There are no markets that exactly match the search criteria.")




Downloaded page 1, total markets: 100
Downloaded page 2, total markets: 200
Downloaded page 3, total markets: 300
Downloaded page 4, total markets: 400
Downloaded page 5, total markets: 500
Downloaded page 6, total markets: 600
Downloaded page 7, total markets: 700
Downloaded page 8, total markets: 800
Downloaded page 9, total markets: 900
Downloaded page 10, total markets: 1000
Downloaded page 11, total markets: 1100
Downloaded page 12, total markets: 1200
Downloaded page 13, total markets: 1300
Downloaded page 14, total markets: 1400
Downloaded page 15, total markets: 1500
Downloaded page 16, total markets: 1600
Downloaded page 17, total markets: 1700
Downloaded page 18, total markets: 1800
Downloaded page 19, total markets: 1900
Downloaded page 20, total markets: 2000
Downloaded page 21, total markets: 2100
Stopping at page=21, offset=10500. Error: 422 Client Error: Unprocessable Entity for url: https://gamma-api.polymarket.com/markets?limit=500&offset=10500&closed=true
Total market

,id,question,endDate_dt,volume,liquidity,yes_token_id
1928,254571,Russian nuke in space in 2024?,2024-12-31 00:00:00+00:00,4.980487e+04,NaN,1149148806850610108009909355425291546655586614...
1935,254578,Will Fed cut interest rates 4 times in 2024?,2024-12-30 12:00:00+00:00,6.812525e+06,NaN,1016691897434389128733611276125893112532020689...
1930,254573,Will Fed cut interest rates 1 time in 2024?,2024-12-30 12:00:00+00:00,1.407495e+06,NaN,3227286084423106081123375705595480745752434016...
1937,254580,Will Fed cut interest rates 6+ times in 2024?,2024-12-30 12:00:00+00:00,2.016909e+07,NaN,8464318340542039420202222890256182674084680835...
1936,254579,Will Fed cut interest rates 5 times in 2024?,2024-12-30 12:00:00+00:00,6.683118e+06,NaN,4729752498249842128346193335853559373400502241...
1929,254572,Will Fed cut interest rates 0 times in 2024?,2024-12-30 12:00:00+00:00,2.049580e+06,NaN,8969665957395422241909525260406718341479132010...
1934,254577,Will Fed cut interest rates 3 times in 2024?,2024-12-30 12:00:00+00:00,8.337157e+06,NaN,3576872514956219269907000630366256701040647233...
1933,254576,Will Fed cut interest rates 2 times in 2024?,2024-12-30 12:00:00+00:00,3.982526e+06,NaN,3871927428485769761941293093301936346966585290...
2003,255337,Fed rate cut by December 18?,2024-12-18 00:00:00+00:00,2.269863e+06,NaN,1143487640821006712994958593113021031697013551...
2002,255336,Fed rate cut by November 7?,2024-11-07 00:00:00+00:00,2.020983e+06,NaN,9322802361841049010117128229226292399515496085...



Selected market:
254578 Will Fed cut interest rates 4 times in 2024?
Price history shape: (6532, 2)


,timestamp,implied_probability
0,2024-03-21 17:00:04+00:00,0.16
1,2024-03-21 18:00:03+00:00,0.16
2,2024-03-21 19:00:03+00:00,0.17
3,2024-03-21 20:00:03+00:00,0.17
4,2024-03-21 21:00:02+00:00,0.17


In [3]:
all_histories = []

for _, row in df_fed.head(5).iterrows():
    end_ts = int(row["endDate_dt"].timestamp())
    start_ts = int((row["endDate_dt"] - pd.Timedelta(days=365)).timestamp())

    hist = get_price_history_chunked(
        clob_token_id=row["yes_token_id"],
        start_ts=start_ts,
        end_ts=end_ts,
        fidelity_minutes=60,
        chunk_days=7
    )

    print(row["id"], row["question"][:70], hist.shape)

    if not hist.empty:
        hist["market_id"] = row["id"]
        hist["question"] = row["question"]
        hist["slug"] = row["slug"]
        all_histories.append(hist)

    time.sleep(0.5)

df_prices_test = pd.concat(all_histories, ignore_index=True)

print(df_prices_test.shape)
display(df_prices_test.head())

248594 Will Hunter Biden be federally indicted by May 1, 2023? (1826, 2)
249778 MLB: Cincinnati Reds vs. Pittsburgh Pirates 2023-04-02 (23, 2)
250474 Did US GDP grow 2.5% or more in Q1 2023? (23, 2)
251025 Will Ron DeSantis file to run for president by May 31, 2023? (212, 2)
251027 USD/TRY (Turkish Lira) above 19.75 on May 22? (146, 2)
(2230, 5)


,timestamp,implied_probability,market_id,question,slug
0,2023-02-07 01:00:32+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
1,2023-02-07 02:00:53+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
2,2023-02-07 03:00:34+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
3,2023-02-07 04:00:05+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
4,2023-02-07 05:00:07+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...


In [4]:
df_prices_test

,timestamp,implied_probability,market_id,question,slug
0,2023-02-07 01:00:32+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
1,2023-02-07 02:00:53+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
2,2023-02-07 03:00:34+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
3,2023-02-07 04:00:05+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
4,2023-02-07 05:00:07+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
...,...,...,...,...,...
2225,2023-05-21 19:00:02+00:00,0.800,251027,USD/TRY (Turkish Lira) above 19.75 on May 22?,usdtry-turkish-lira-above-19pt75-on-may-22
2226,2023-05-21 20:00:02+00:00,0.850,251027,USD/TRY (Turkish Lira) above 19.75 on May 22?,usdtry-turkish-lira-above-19pt75-on-may-22
2227,2023-05-21 21:00:02+00:00,0.825,251027,USD/TRY (Turkish Lira) above 19.75 on May 22?,usdtry-turkish-lira-above-19pt75-on-may-22
2228,2023-05-21 22:00:02+00:00,0.825,251027,USD/TRY (Turkish Lira) above 19.75 on May 22?,usdtry-turkish-lira-above-19pt75-on-may-22
